# 06 — Memory across runs
Flow state belongs to one run. Memory can be recalled in a later run.
Explicit scopes keep Customer A's preference out of Customer B's context.



**Basic lesson 6 of 13.**


## Problem

A new run does not automatically know a customer's saved constraint.


## Concept

CrewAI Memory recalls retained facts; explicit scopes separate customers.

Memory retains selected facts across invocations. A scoped recall narrows what is eligible to be returned, so customer identity must be chosen by the application, never guessed from the prompt.


## Where this fits in the whole workshop

```text
USER · CLI · API
   │
   ▼
 13 PROJECT        run_assistant(): memory in → Flow → report out
   │
   ├─ 06 MEMORY          recall this customer's saved constraint   ◀ YOU ARE HERE
   ├─ 09 CONTROL         validate the question before any model call
   ├─ 08 FLOW            typed state · route · one bounded retry
   │    └─ 07 CREW            Researcher → Analyst → Critic
   │         └─ 02 AGENT LOOP      model decides → tool runs → observation → answer
   │              ├─ 01 MODEL          the LLM call itself
   │              ├─ 03 TOOLS          typed Python functions the agent may call
   │              ├─ 04 MCP            tools served by a separate process
   │              └─ 05 KNOWLEDGE      cited passages from company documents
   └─ 09 CONTROL         validate the report → human approval → mock email

 around everything:
   10 RESILIENCE      checkpoint and resume
   11 OBSERVABILITY   trace what actually ran
   12 EVALUATION      checks that fail loudly
```

Every lesson adds one row to this system. The rows below the marker are still the same code from earlier lessons.


## Architecture of this lesson

```text
customer A fact ──▶ /customer/customer-a ──┐
                                           ├──▶ scoped recall ──▶ next run
customer B fact ──▶ /customer/customer-b ──┘
                  (B's facts must not enter A's context)
```

customer A/B facts → scoped store → scoped recall in a later run. A tool result or validation failure is an observation, not an instruction the model may bypass.


## Step 1 — Create Memory and save two customers' facts in distinct scopes

Create Memory and save two customers' facts in distinct scopes.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[2] if "__file__" in globals() else Path.cwd()))

import workshop  # configure project-local CrewAI storage
from crewai import Memory

from workshop.common import embedder_config
from workshop.memory import customer_scope

memory = Memory(embedder=embedder_config())
memory.remember(
    "Customer A requires private VPC deployment.",
    scope=customer_scope("customer-a"),
    source="customer:customer-a",
)
memory.remember(
    "Customer B prefers a managed SaaS pilot.",
    scope=customer_scope("customer-b"),
    source="customer:customer-b",
)



## Recall from a customer scope
Inspect both results. A result from the wrong customer indicates a serious
isolation problem, not just a weak answer.



## Step 2 — Recall each scope independently and inspect the tree for isolation

Recall each scope independently and inspect the tree for isolation.


In [ ]:
for customer in ("customer-a", "customer-b"):
    matches = memory.recall("deployment preference", scope=customer_scope(customer), limit=3)
    print(customer, [match.record.content for match in matches])
print("Memory tree:\n", memory.tree())



## Bonus scenario — team meeting notes
Scopes work for any grouping, here teams (inspired by CrewAI's *Meeting Assistant* example).
Each team's action item stays in its own scope.



In [ ]:
memory.remember("Design team: send the new logo files to marketing by Friday.", scope="/team/design")
memory.remember("Data team: retrain the churn model after the schema change.", scope="/team/data")
for team in ("design", "data"):
    matches = memory.recall("what is the action item?", scope=f"/team/{team}", limit=2)
    print(team, [match.record.content for match in matches])


## Failure mode to know about

Using a global scope leaks one customer's fact into another customer's run. The scope identifier is a security boundary, not a personalization convenience.


## Production implication

Define what may be retained, for how long, and how it is deleted. Test tenant isolation with negative cases, not just successful recall.


## Your turn

Search both scopes for deployment preferences and inspect the memory tree.

**Checkpoint:** A's VPC preference does not appear in B's recall.
